# Identity Naming — task demo

Three sections:

1. **Causal model** — the entity→result lookup.
2. **Templates & token positions** — where the entity token lands in the prompt.
3. **Counterfactual generators** — how `generate_dataset` cycles through phrasings.

Tokenization uses `gpt2`. No interventions — see `analyses/locate/demo.ipynb` for those.

Currently one domain ships: **`pitch_midi`** (musical note name → MIDI number).

In [1]:
from causalab.tasks.identity_naming import (
    IdentityNamingConfig,
    create_causal_model,
    create_token_positions,
    generate_dataset,
)

cfg = IdentityNamingConfig(domain_type="pitch_midi")
model = create_causal_model(cfg)
(model.id, len(cfg.entities), cfg.entities[:5], cfg.entities[-5:])

('identity_naming_pitch_midi',
 49,
 ['C2', 'C#2', 'D2', 'D#2', 'E2'],
 ['G#5', 'A5', 'A#5', 'B5', 'C6'])

## 1. Causal model

The DAG is `entity → result → raw_output` plus `entity → raw_input`. `result` is a dictionary lookup (`entity_to_result[entity]`), not arithmetic. Sample a few entities and trace them.

In [2]:
import random

random.seed(0)
for _ in range(4):
    trace = model.sample_input()
    print(
        f"  entity={trace['entity']:<5} -> result={trace['result']:<3} | "
        f"raw_input={trace['raw_input']!r}  raw_output={trace['raw_output']!r}"
    )

  entity=C4    -> result=60  | raw_input='The MIDI number for C4 is '  raw_output='60'
  entity=C6    -> result=84  | raw_input='The MIDI number for C6 is '  raw_output='84'
  entity=D4    -> result=62  | raw_input='The MIDI number for D4 is '  raw_output='62'
  entity=D2    -> result=38  | raw_input='The MIDI number for D2 is '  raw_output='38'


Note the result is the canonical MIDI number for the note. Range is `C2` (MIDI 36) through `C6` (MIDI 84) — the central piano range. The result is embedded by integer value (via `_pitch_midi_result_embed`), so geometry analyses interpret distances as semitone steps.

In [3]:
embeddings = model.embeddings
print("entity 'C#4'      ->", embeddings["entity"]("C#4"))
print("entity 'D4'       ->", embeddings["entity"]("D4"))
print("result '61'       ->", embeddings["result"]("61"))
print("result '62'       ->", embeddings["result"]("62"))

entity 'C#4'      -> [61.0]
entity 'D4'       -> [62.0]
result '61'       -> [61.0]
result '62'       -> [62.0]


## 2. Templates & token positions

`create_token_positions(pipeline, template=...)` returns `last_token` and `entity` positions. The `entity` position locates the last token spanning the `{entity}` slot via the declarative `scope` mechanism (no per-model tokenization tables).

In [4]:
from causalab.neural.pipeline import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=1)
positions = create_token_positions(pipeline, template=cfg.templates[0])
list(positions.keys())

`torch_dtype` is deprecated! Use `dtype` instead!


['last_token', 'entity']

In [5]:
trace = model.sample_input()
ids = pipeline.load([trace])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]
pad_id = pipeline.tokenizer.pad_token_id

print("Prompt:", trace["raw_input"])
print()
print(f"{'idx':>4}  {'token':<20}  positions")
for i, tok in enumerate(decoded):
    if ids[i] == pad_id:
        continue
    hits = [name for name, pos in positions.items() if i in pos.index(trace)]
    marker = ", ".join(hits) if hits else ""
    print(f"{i:>4}  {tok!r:<20}  {marker}")

Prompt: The MIDI number for E3 is 

 idx  token                 positions
   0  'The'                 
   1  ' MIDI'               
   2  ' number'             
   3  ' for'                
   4  ' E'                  
   5  '3'                   entity
   6  ' is'                 
   7  ' '                   last_token


## 3. Counterfactual generators

`generate_dataset` cycles through the preset's `templates` list and returns `n` pairs `{"input": trace, "counterfactual_inputs": [trace]}`. Within a pair, both base and counterfactual share the same template (so phrasing isn't a per-pair confound), but they sample independent entities. Across the dataset, examples cycle through templates to provide phrasing variation for centroid computation downstream.

`pitch_midi` ships with one template, so cycling is trivial here — but the same code handles multi-template presets without changes.

In [6]:
for i, ex in enumerate(generate_dataset(model, n=4, seed=0)):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"--- pair {i} ---")
    print(
        f"  base : entity={base['entity']:<5} -> result={base['result']:<3}  raw_input={base['raw_input']!r}"
    )
    print(
        f"  cf   : entity={cf['entity']:<5} -> result={cf['result']:<3}    raw_input={cf['raw_input']!r}"
    )
    print()

--- pair 0 ---
  base : entity=C4    -> result=60   raw_input='The MIDI number for C4 is '
  cf   : entity=C6    -> result=84     raw_input='The MIDI number for C6 is '

--- pair 1 ---
  base : entity=D4    -> result=62   raw_input='The MIDI number for D4 is '
  cf   : entity=D2    -> result=38     raw_input='The MIDI number for D2 is '

--- pair 2 ---
  base : entity=E3    -> result=52   raw_input='The MIDI number for E3 is '
  cf   : entity=G#4   -> result=68     raw_input='The MIDI number for G#4 is '

--- pair 3 ---
  base : entity=G4    -> result=67   raw_input='The MIDI number for G4 is '
  cf   : entity=C#4   -> result=61     raw_input='The MIDI number for C#4 is '



## Next steps

Compose a runner config that mounts `task: identity_naming_pitch_midi` and the analyses you need (e.g. `analysis/baseline`, `analysis/locate`). See `causalab/configs/task/identity_naming_pitch_midi.yaml` for the task-side defaults; `isometry.grid_range: [36, 84]` is set to the MIDI range so geometry analyses see distances in semitone units.